# Classificação binária com EfficientNet (Demented vs NonDemented)

In [4]:
# ===== 1) Setup e Config =====
import os, glob, random, math
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import efficientnet

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

# Caminhos
# Resolve o diretório raiz do projeto para que 'database' seja encontrado
ROOT = Path('.')
for candidate in [Path('.'), Path('..'), Path('../..')]:
    if (candidate / 'database').exists():
        ROOT = candidate
        break
DB_ROOT = ROOT / 'database'
AXL_DIR = DB_ROOT / 'axl'  # Arquivos .nii.gz com padrão {MRI ID}_axl.nii.gz
TRAIN_DIR = DB_ROOT / 'treino'
TEST_DIR  = DB_ROOT / 'teste'

# CSVs gerados pelo db_split.ipynb
TRAIN_OASIS_CSV = TRAIN_DIR / 'oasis_longitudinal_demographic.csv'
TRAIN_IDS_CSV   = TRAIN_DIR / 'features_identifiers.csv'
TEST_OASIS_CSV  = TEST_DIR / 'oasis_longitudinal_demographic.csv'
TEST_IDS_CSV    = TEST_DIR / 'features_identifiers.csv'

# Hiperparâmetros
IMG_SIZE   = 224
BATCH_SIZE = 8
EPOCHS     = 25
VAL_SPLIT  = 0.15
LR_BASE    = 1e-4
AUGMENT    = True
MODEL_VARIANT = 'B0'  # B0/B1/B2...

# Mapeamento de classes
CLASS_MAP = {'NonDemented': 0, 'Demented': 1}

TF version: 2.20.0
GPUs: []


In [ ]:
# ===== 2) Carregar splits e montar tabela de caminhos =====
def read_split(oasis_csv, ids_csv):
    df_oasis = pd.read_csv(oasis_csv, sep=';', decimal=',')
    df_ids   = pd.read_csv(ids_csv,   sep=';', decimal=',')

    # Normaliza classes
    df_oasis['Group'] = df_oasis['Group'].replace({'Nondemented': 'NonDemented'})
    df_oasis = df_oasis[df_oasis['Group'].isin(['Demented', 'NonDemented'])].copy()

    # Mapear label binário por MRI ID
    df_oasis['y'] = df_oasis['Group'].map(CLASS_MAP)
    df_mri = df_oasis[['MRI ID', 'y']].drop_duplicates()  # um rótulo por MRI

    # Junta com identificadores
    df = pd.merge(df_mri, df_ids, on='MRI ID', how='left')
    return df

df_train = read_split(TRAIN_OASIS_CSV, TRAIN_IDS_CSV)
df_test  = read_split(TEST_OASIS_CSV,  TEST_IDS_CSV)

print('Train MRI IDs:', len(df_train), 'Test MRI IDs:', len(df_test))
display(df_train.head())

GLOB_PATTERNS = [
    "{mri}_axl.nii.gz",
]

def find_images_for_mri(mri_id):
    paths = []
    for pat in GLOB_PATTERNS:
        gl = str(AXL_DIR / pat.format(mri=mri_id))
        paths.extend(glob.glob(gl))
    # Remove duplicadas mantendo ordem
    seen = set()
    uniq = []
    for p in paths:
        if p not in seen:
            uniq.append(p)
            seen.add(p)
    return uniq

def expand_to_paths(df):
    rows = []
    missing = []
    for mri_id, y in df[['MRI ID','y']].itertuples(index=False):
        vols = find_images_for_mri(mri_id)
        if not vols:
            missing.append(mri_id)
            continue
        # usamos um volume por MRI (se houver mais de um match, pegamos o primeiro)
        p = vols[0]
        rows.append((p, y, mri_id))
    out = pd.DataFrame(rows, columns=['path','y','mri'])
    return out, missing

train_paths, miss_train = expand_to_paths(df_train)
test_paths,  miss_test  = expand_to_paths(df_test)


print(f"Train volumes: {len(train_paths)} | MRIs sem volume: {len(miss_train)}")
print(f"Test  volumes: {len(test_paths)}  | MRIs sem volume: {len(miss_test)}")
if miss_train or miss_test:
    print('Exemplos de MRIs sem volume:', (miss_train[:5] + miss_test[:5]))

display(train_paths.head())

Train MRI IDs: 143 Test MRI IDs: 40


,MRI ID,y,Subject ID,Group,Ventricle_Area,Ventricle_Perimeter,Ventricle_Circularity,Ventricle_Eccentricity,Ventricle_Solidity,Ventricle_MajorAxisLength
0,OAS2_0002_MR1,1,OAS2_0002,Demented,4809.0,870.264069,0.079793,0.776428,0.502602,117.960323
1,OAS2_0002_MR2,1,OAS2_0002,Demented,4566.0,609.819372,0.154292,0.793689,0.528488,127.950254
2,OAS2_0002_MR3,1,OAS2_0002,Demented,5744.0,823.979797,0.106314,0.776788,0.565456,123.596244
3,OAS2_0010_MR1,1,OAS2_0010,Demented,2659.0,554.671140,0.108607,0.936848,0.680314,93.539359
4,OAS2_0010_MR2,1,OAS2_0010,Demented,2923.0,599.535101,0.102190,0.952558,0.644067,97.121640


Train volumes: 143 | MRIs sem volume: 0
Test  volumes: 40  | MRIs sem volume: 0


,path,y,mri
0,..\database\axl\OAS2_0002_MR1_axl.nii.gz,1,OAS2_0002_MR1
1,..\database\axl\OAS2_0002_MR2_axl.nii.gz,1,OAS2_0002_MR2
2,..\database\axl\OAS2_0002_MR3_axl.nii.gz,1,OAS2_0002_MR3
3,..\database\axl\OAS2_0010_MR1_axl.nii.gz,1,OAS2_0010_MR1
4,..\database\axl\OAS2_0010_MR2_axl.nii.gz,1,OAS2_0010_MR2


In [6]:
# ===== 3) Dataset TF: leitura NIfTI, extração de slice e augment =====
AUTOTUNE = tf.data.AUTOTUNE

def load_nifti_slice_np(path_str, mode='center', jitter=2):
    import numpy as np
    import nibabel as nib
    img_nii = nib.load(path_str)
    vol = img_nii.get_fdata()
    if vol.ndim == 4:
        vol = vol[..., 0]
    zdim = vol.shape[-1]
    idx = zdim // 2
    if mode == 'random' and zdim > 5:
        low = max(0, idx - jitter)
        high = min(zdim - 1, idx + jitter)
        idx = np.random.randint(low, high + 1)
    sl = vol[..., idx]
    # Normalização robusta
    p2, p98 = np.percentile(sl, (2, 98))
    if p98 > p2:
        sl = np.clip((sl - p2) / (p98 - p2), 0, 1)
    else:
        mn, mx = sl.min(), sl.max()
        sl = (sl - mn) / (mx - mn + 1e-6)
    # 3 canais
    sl = np.stack([sl, sl, sl], axis=-1).astype(np.float32)
    return sl

def decode_nifti(path, training):
    def _loader(p):
        p_str = p.numpy().decode('utf-8')
        return load_nifti_slice_np(p_str, mode=('random' if training else 'center'))
    img = tf.py_function(func=_loader, inp=[path], Tout=tf.float32)
    img.set_shape([None, None, 3])
    img4 = tf.expand_dims(img, axis=0)
    img4 = tf.image.resize(img4, (IMG_SIZE, IMG_SIZE), method='bilinear')
    img = tf.squeeze(img4, axis=0)
    img = tf.ensure_shape(img, (IMG_SIZE, IMG_SIZE, 3))
    return img

def make_dataset(df_paths, training=False, shuffle=True):
    x = df_paths['path'].values
    y = df_paths['y'].values.astype('int32')

    ds_x = tf.data.Dataset.from_tensor_slices(x)
    ds_y = tf.data.Dataset.from_tensor_slices(y)
    ds   = tf.data.Dataset.zip((ds_x, ds_y))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df_paths), 1024), seed=SEED, reshuffle_each_iteration=True)

    def _map(path, label):
        img = decode_nifti(path, training=training)
        return img, tf.cast(label, tf.int32)

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)

    if training and AUGMENT:
        aug = keras.Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.05),
            layers.RandomZoom(0.1, 0.1),
        ])
        ds = ds.map(lambda img, label: (aug(img, training=True), label), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Split treino/val por volume
val_count = max(1, int(len(train_paths) * VAL_SPLIT))
train_df = train_paths.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df   = train_df.iloc[:val_count].reset_index(drop=True)
train_df = train_df.iloc[val_count:].reset_index(drop=True)

ds_train = make_dataset(train_df, training=True, shuffle=True)
ds_val   = make_dataset(val_df,   training=False, shuffle=False)
ds_test  = make_dataset(test_paths, training=False, shuffle=False)

len_train_steps = math.ceil(len(train_df) / BATCH_SIZE)
len_val_steps   = math.ceil(len(val_df)   / BATCH_SIZE)
len_test_steps  = math.ceil(len(test_paths)/ BATCH_SIZE)
print('steps -> train/val/test:', len_train_steps, len_val_steps, len_test_steps)

steps -> train/val/test: 16 3 5


In [10]:
# ===== 4) Modelo EfficientNetB0 e compilação =====
def build_model(num_classes=1):
    base = efficientnet.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False  # primeiro: congela base

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = inputs
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(LR_BASE),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ]
    )
    return model

model = build_model()
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [11]:
# ===== 5) Treinamento com callbacks e fine-tuning =====
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_auc', mode='max'),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, min_lr=1e-6, monitor='val_auc', mode='max'),
]

# history com ou sem class_weight (descomente bloco de class_weight para usar)
# counts = train_df['y'].value_counts().to_dict()
# total = sum(counts.values())
# class_weight = {c: total/(2*cnt) for c, cnt in counts.items()}
# print('class_weight:', class_weight)
# history = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, callbacks=callbacks, class_weight=class_weight, verbose=1)

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# Unfreeze parcial (fine-tuning)
base_layer = None
for l in model.layers:
    if isinstance(l, keras.Model) and 'efficientnet' in l.name:
        base_layer = l
        break
if base_layer is None:
    try:
        base_layer = model.get_layer('efficientnetb0')
    except Exception:
        base_layer = None

if base_layer is not None:
    for layer in base_layer.layers:
        layer.trainable = True
    unfreeze_from = 200
    for layer in base_layer.layers[:unfreeze_from]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(LR_BASE * 0.1),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
    )
    history_ft = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=max(10, EPOCHS//2),
        callbacks=callbacks,
        verbose=1
    )
else:
    print('Aviso: camada base EfficientNet não encontrada para fine-tuning.')

Epoch 1/25


InvalidArgumentError: Graph execution error:

Detected at node resize/ResizeBilinear defined at (most recent call last):
<stack traces unavailable>
Error in user-defined function passed to ParallelMapDatasetV2:4 transformation with iterator: Iterator::Root::Prefetch::MapAndBatch::ParallelMapV2: input must be 4-dimensional[1,207,3]
	 [[{{node resize/ResizeBilinear}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_34936]

In [ ]:
# ===== 6) Avaliação por volume (slice central) =====
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

for imgs, labels in ds_test:
    probs = model.predict(imgs, verbose=0).ravel()
    y_pred.extend((probs >= 0.5).astype(int))
    y_true.extend(labels.numpy().astype(int))

print(classification_report(y_true, y_pred, target_names=['NonDemented','Demented']))
print(confusion_matrix(y_true, y_pred))

In [ ]:
# ===== 7) (Opcional) Média de probabilidades por MRI (se houvesse múltiplos slices) =====
# Como usamos 1 volume -> 1 slice por amostra, a agregação por MRI
# não muda o resultado. Mantido aqui para compatibilidade.
def make_dataset_from_subdf(subdf):
    return make_dataset(subdf.reset_index(drop=True), training=False, shuffle=False)

def predict_by_mri(df_paths):
    grouped = df_paths.groupby('mri')
    probs_by_mri = {}
    labels_by_mri = {}
    for mri, sub in grouped:
        ds = make_dataset_from_subdf(sub)
        probs = []
        lab = None
        for imgs, labels in ds:
            pr = model.predict(imgs, verbose=0).ravel()
            probs.extend(pr.tolist())
            if lab is None:
                lab = int(labels[0].numpy())
        probs_by_mri[mri] = float(np.mean(probs)) if probs else 0.0
        labels_by_mri[mri] = lab if lab is not None else 0
    return probs_by_mri, labels_by_mri

probs_mri, labels_mri = predict_by_mri(test_paths)
y_true_mri = []
y_pred_mri = []
for mri, p in probs_mri.items():
    y_true_mri.append(labels_mri[mri])
    y_pred_mri.append(int(p >= 0.5))

print('Avaliação por MRI:')
print(classification_report(y_true_mri, y_pred_mri, target_names=['NonDemented','Demented']))
print(confusion_matrix(y_true_mri, y_pred_mri))